In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Result files in outputs/results/ with their paper labels (Table 2 order)
MODELS = {
    "statistical-baseline.json":  "Statistical Baseline",
    "ministral-3-3b.json":        "Ministral 3 3B",
    "ministral-3-3b-sft.json":    "Ministral 3 3B (SFT)",
    "ministral-3-8b.json":        "Ministral 3 8B",
    "ministral-3-8b-sft.json":    "Ministral 3 8B (SFT)",
    "gemini-3.5-flash-lite.json": "Gemini 3.5 Flash Lite",  # accuracy only (no logits)
}


In [ ]:
def load_history_data(results_dir: str):
    """Loads all JSON files, applies OOD normalization, and extracts history lengths."""
    
    all_data = []
    
    for filename, model_name in MODELS.items():
        filepath = os.path.join(results_dir, filename)
        if not os.path.exists(filepath):
            continue
        
        with open(filepath, "r") as f:
            results = json.load(f)
            
        for r in results:
            probs = r.get("probabilities")  # absent for the proprietary baseline
            p0 = probs.get("0", 0.5) if probs else 0.0
            p1 = probs.get("1", 0.5) if probs else 0.0
            total_valid_mass = p0 + p1
            
            # 1. Renormalize to factor out formatting noise (Methodological Continuity)
            if not probs:
                conf = np.nan
            elif total_valid_mass > 0:
                norm_p0 = p0 / total_valid_mass
                norm_p1 = p1 / total_valid_mass
                conf = max(norm_p0, norm_p1)
            else:
                conf = 0.5 # Fallback to maximum uncertainty if 100% OOD
                
            # 2. Extract accuracy and history length
            acc = 1 if str(r["true_label"]) == str(r["prediction"]) else 0
            hl = r.get("history_length", 3) # Defaults to 3 if missing for some reason
            
            all_data.append({
                "Model": model_name,
                "History Length": hl,
                "Accuracy": acc,
                "Confidence": conf
            })
            
    return pd.DataFrame(all_data)

In [ ]:
import pandas as pd
from scipy.stats import linregress, pearsonr

def analyze_contextual_scaling(df: pd.DataFrame):
    """
    Calculates gradients, correlations, and p-values for contextual scaling.
    Groups by History Length to analyze the macroscopic trends.
    """
    metrics = []
    
    # Group by Model and History Length to analyze the scaling trends
    grouped = df.groupby(["Model", "History Length"]).mean().reset_index()
    
    def format_p(p_val):
        """Helper to format p-values cleanly for academic tables."""
        if np.isnan(p_val):
            return "-"
        if p_val < 0.001:
            return "<0.001"
        return f"{p_val:.3f}"
    
    for model_key in [m for m in MODELS.values() if m in set(grouped['Model'])]:
        model_data = grouped[grouped['Model'] == model_key]
        
        # 1. Scaling of Accuracy vs History Length
        slope_acc, _, r_acc, p_acc, _ = linregress(model_data["History Length"], model_data["Accuracy"])
        
        # 2. Scaling of Confidence vs History Length
        # (skipped when no token probabilities are available)
        if model_data["Confidence"].notna().all():
            slope_conf, _, r_conf, p_conf, _ = linregress(model_data["History Length"], model_data["Confidence"])
            
            # 3. Direct Correlation: Accuracy vs Confidence
            r_ac, p_ac = pearsonr(model_data["Accuracy"], model_data["Confidence"])
        else:
            slope_conf = r_conf = p_conf = r_ac = p_ac = np.nan
        
        metrics.append({
            "Model": model_key,
            "Δ Acc": slope_acc * 100, 
            "r_acc": r_acc, "p_acc": format_p(p_acc),
            "Δ Conf": slope_conf * 100, 
            "r_conf": r_conf, "p_conf": format_p(p_conf),
            "r_ac": r_ac, "p_ac": format_p(p_ac)
        })

    df_metrics = pd.DataFrame(metrics)
    
    # --- Generate LaTeX Table ---
    latex = []
    latex.append("\\begin{table*}[t]")
    latex.append("\\centering")
    # Using booktabs formatting with a slightly wider stance for the new columns
    latex.append("\\begin{tabular}{@{}l|cc|cc|cc@{}}")
    latex.append("\\toprule")
    latex.append("\\textbf{Model} & \\multicolumn{2}{c|}{\\textbf{History $\\to$ Acc}} & \\multicolumn{2}{c|}{\\textbf{History $\\to$ Conf}} & \\multicolumn{2}{c}{\\textbf{Acc $\\leftrightarrow$ Conf}} \\\\")
    latex.append("\\cmidrule(lr){2-3} \\cmidrule(lr){4-5} \\cmidrule(lr){6-7}")
    latex.append(" & $\\Delta_{100}$ & $r$ ($p$) & $\\Delta_{100}$ & $r$ ($p$) & $r$ & $p$ \\\\")
    latex.append("\\midrule")
    
    for _, row in df_metrics.iterrows():
        m_name = row["Model"]
        
        # Formatting with p-values in parentheses
        str_acc = f"{row['Δ Acc']:.3f} & {row['r_acc']:.2f} ({row['p_acc']})"
        if np.isnan(row["Δ Conf"]):
            str_conf, str_ac = "- & -", "- & -"
        else:
            str_conf = f"{row['Δ Conf']:.3f} & {row['r_conf']:.2f} ({row['p_conf']})"
            str_ac = f"{row['r_ac']:.2f} & {row['p_ac']}"
        
        latex.append(f"{m_name} & {str_acc} & {str_conf} & {str_ac} \\\\")
        
    latex.append("\\bottomrule")
    latex.append("\\end{tabular}")
    latex.append("\\caption{\\textbf{Contextual Scaling.} "
                 "$\\Delta_{100}$ represents the linear change in probability mass per 100 interactions. "
                 "The Acc $\\leftrightarrow$ Conf correlation tests the contextual overconfidence hypothesis.}")
    latex.append("\\label{tab:context_scaling}")
    latex.append("\\end{table*}")
    
    print("\n".join(latex))

In [ ]:
df_history = load_history_data("../../outputs/results/")

analyze_contextual_scaling(df_history)